# **TA-3.1 – Patrones MapReduce – Estudiante: Savier Torres Soto**
**Proyecto Integrador**
Predicción de Ventas Minoristas enfocado en la Coorporacion La Favorita


In [3]:
!pip install pyspark --quiet

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    count,
    sum as spark_sum,
    avg,
    max as spark_max,
    min as spark_min,
    round as spark_round,
    row_number
)

from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("TA3.1-Patrones-MapReduce") \
    .master("local[*]") \
    .getOrCreate()

sc = spark.sparkContext

print("✓ Spark iniciado")
print(f"Versión: {spark.version}")
print(f"Cores disponibles: {sc.defaultParallelism}")

✓ Spark iniciado
Versión: 4.0.2
Cores disponibles: 2


# **Carga e integración de los datos**

El proyecto utiliza el dataset de predicción de ventas minoristas de Corporación Favorita. Debido a que la información se encuentra distribuida en varios archivos, se realiza un proceso de integración para construir un dataset maestro que será utilizado en los análisis MapReduce.

In [5]:
train_df = spark.read.csv(
    "train.csv",
    header=True,
    inferSchema=True
)

stores_df = spark.read.csv(
    "stores.csv",
    header=True,
    inferSchema=True
)

oil_df = spark.read.csv(
    "oil.csv",
    header=True,
    inferSchema=True
)

df_master = train_df.join(
    stores_df,
    on="store_nbr",
    how="left"
).join(
    oil_df,
    on="date",
    how="left"
)

print(f"\n✓ Dataset cargado: {df_master.count():,} filas × {len(df_master.columns)} columnas")

df_master.printSchema()
df_master.show(5)


✓ Dataset cargado: 3,000,888 filas × 11 columnas
root
 |-- date: date (nullable = true)
 |-- store_nbr: integer (nullable = true)
 |-- id: integer (nullable = true)
 |-- family: string (nullable = true)
 |-- sales: double (nullable = true)
 |-- onpromotion: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- type: string (nullable = true)
 |-- cluster: integer (nullable = true)
 |-- dcoilwtico: double (nullable = true)

+----------+---------+---+----------+-----+-----------+-----+---------+----+-------+----------+
|      date|store_nbr| id|    family|sales|onpromotion| city|    state|type|cluster|dcoilwtico|
+----------+---------+---+----------+-----+-----------+-----+---------+----+-------+----------+
|2013-01-01|        1|  0|AUTOMOTIVE|  0.0|          0|Quito|Pichincha|   D|     13|      NULL|
|2013-01-01|        1|  1| BABY CARE|  0.0|          0|Quito|Pichincha|   D|     13|      NULL|
|2013-01-01|        1|  2|    BEAUTY|  0.0|

# **Patrón 1 – Conteo por Categoría**
**Pregunta de negocio**

**¿Cuántos registros de ventas existen para cada familia de productos?**
Este análisis permite identificar las categorías con mayor presencia dentro de la operación comercial y conocer qué líneas de productos tienen una participación más significativa en el conjunto de datos.

In [6]:
conteo_familias = df_master.groupBy("family") \
                           .count() \
                           .orderBy(col("count").desc())

conteo_familias.show(20, False)

+-------------------+-----+
|family             |count|
+-------------------+-----+
|PREPARED FOODS     |90936|
|HOME AND KITCHEN II|90936|
|LADIESWEAR         |90936|
|LAWN AND GARDEN    |90936|
|GROCERY I          |90936|
|BABY CARE          |90936|
|PRODUCE            |90936|
|AUTOMOTIVE         |90936|
|BEVERAGES          |90936|
|HOME CARE          |90936|
|BREAD/BAKERY       |90936|
|BOOKS              |90936|
|LINGERIE           |90936|
|CELEBRATION        |90936|
|GROCERY II         |90936|
|DAIRY              |90936|
|MAGAZINES          |90936|
|SEAFOOD            |90936|
|LIQUOR,WINE,BEER   |90936|
|HOME AND KITCHEN I |90936|
+-------------------+-----+
only showing top 20 rows


# **Interpretación**

Los resultados muestran cuántos registros existen para cada familia de productos.
La categoría que aparezca en la primera posición representa una de las líneas de negocio más importantes dentro del dataset, ya que concentra la mayor cantidad de registros de ventas.

Esta información ayuda a comprender la distribución de los productos y permite identificar qué categorías tienen mayor relevancia para futuros análisis y modelos predictivos.

# **Patrón 2 – Agregación Numérica Múltiple**
**Pregunta de negocio**

**¿Cómo se comportan las ventas según la tienda y la familia de productos?**

Para responder esta pregunta se calculan las ventas totales, promedio y máximas agrupadas por tienda y categoría de producto.

In [7]:
agregacion_ventas = df_master.groupBy(
    "store_nbr",
    "family"
).agg(
    spark_sum("sales").alias("ventas_totales"),
    avg("sales").alias("ventas_promedio"),
    spark_max("sales").alias("venta_maxima")
)

agregacion_ventas.orderBy(
    col("ventas_totales").desc()
).show(20, False)

+---------+---------+--------------------+------------------+------------+
|store_nbr|family   |ventas_totales      |ventas_promedio   |venta_maxima|
+---------+---------+--------------------+------------------+------------+
|44       |GROCERY I|1.6386055037999999E7|9730.436483372921 |46271.0     |
|45       |GROCERY I|1.6349751291E7      |9708.878438836104 |76090.0     |
|47       |GROCERY I|1.551452849E7       |9212.902903800476 |23024.0     |
|46       |GROCERY I|1.4342262E7         |8516.782660332541 |25238.0     |
|44       |BEVERAGES|1.3417859E7         |7967.849762470309 |25413.0     |
|3        |GROCERY I|1.2970467971E7      |7702.178130047507 |21858.0     |
|48       |GROCERY I|1.2830994111E7      |7619.35517280285  |22255.0     |
|45       |BEVERAGES|1.1370352E7         |6751.990498812352 |22170.0     |
|3        |BEVERAGES|1.1351589E7         |6740.848574821853 |19154.0     |
|49       |GROCERY I|1.108798E7          |6584.311163895487 |21190.0     |
|11       |GROCERY I|1.10

# **Interpretación**
Las ventas totales permiten identificar las combinaciones de tienda y categoría que generan mayores ingresos.
El promedio de ventas muestra el comportamiento habitual de cada categoría, mientras que la venta máxima permite detectar picos de demanda.

Estos resultados son útiles para optimizar inventarios, planificar promociones y mejorar la distribución de productos en las distintas tiendas.

# **Patrón 3 – Ranking con Window Function**
**Pregunta de negocio**

**¿Cuál es la familia de productos con mayores ventas dentro de cada tienda?**

Este análisis permite identificar la categoría líder en cada establecimiento.

In [8]:
ventas_familia = df_master.groupBy(
    "store_nbr",
    "family"
).agg(
    spark_sum("sales").alias("ventas_totales")
)

window_spec = Window.partitionBy(
    "store_nbr"
).orderBy(
    col("ventas_totales").desc()
)

ranking = ventas_familia.withColumn(
    "ranking",
    row_number().over(window_spec)
)

top_categoria_por_tienda = ranking.filter(
    col("ranking") == 1
)

top_categoria_por_tienda.show(50, False)

+---------+---------+--------------------+-------+
|store_nbr|family   |ventas_totales      |ranking|
+---------+---------+--------------------+-------+
|1        |GROCERY I|3743823.0           |1      |
|2        |GROCERY I|6591703.797         |1      |
|3        |GROCERY I|1.2970467971E7      |1      |
|4        |GROCERY I|5636343.59          |1      |
|5        |GROCERY I|5262681.658         |1      |
|6        |GROCERY I|7806646.169         |1      |
|7        |GROCERY I|6247973.995999999   |1      |
|8        |GROCERY I|7909966.0           |1      |
|9        |GROCERY I|1.0353098959999993E7|1      |
|10       |GROCERY I|4126220.1739999996  |1      |
|11       |GROCERY I|1.1044948458000004E7|1      |
|12       |GROCERY I|4208523.907999999   |1      |
|13       |GROCERY I|4570080.779000001   |1      |
|14       |GROCERY I|4463305.511         |1      |
|15       |GROCERY I|4578222.107999999   |1      |
|16       |GROCERY I|4307272.248000002   |1      |
|17       |GROCERY I|6453635.21

# **Interpretación**
El ranking identifica la categoría con mejores resultados de ventas dentro de cada tienda.

Este hallazgo permite conocer las preferencias predominantes de los clientes según la ubicación y facilita la toma de decisiones relacionadas con inventario, abastecimiento y estrategias comerciales específicas para cada establecimiento.

# **Patrón Opcional – Filtro + Agregación**
**Pregunta de negocio**

**¿Qué categorías generan más ventas cuando existen promociones activas?**

Este análisis permite evaluar el impacto de las promociones sobre las ventas.

In [9]:
promociones = df_master.filter(
    col("onpromotion") > 0
)

ventas_promocion = promociones.groupBy(
    "family"
).agg(
    spark_sum("sales").alias("ventas_totales"),
    avg("sales").alias("ventas_promedio")
)

ventas_promocion.orderBy(
    col("ventas_totales").desc()
).show(20, False)

+--------------------------+--------------------+------------------+
|family                    |ventas_totales      |ventas_promedio   |
+--------------------------+--------------------+------------------+
|GROCERY I                 |2.5092434118899894E8|4411.003431230864 |
|BEVERAGES                 |1.6625734E8         |3215.498307707185 |
|PRODUCE                   |7.505817053904594E7 |2438.2201968245176|
|CLEANING                  |6.1886516E7         |1240.210741482966 |
|DAIRY                     |4.2919719E7         |933.341720126128  |
|BREAD/BAKERY              |2.2384076085149966E7|575.4852963068173 |
|PERSONAL CARE             |1.2950625E7         |338.74669770605004|
|DELI                      |1.2699044140076008E7|321.6658005541176 |
|HOME CARE                 |1.0068717E7         |309.44486446616264|
|MEATS                     |1.0032031411536999E7|467.3016308709241 |
|POULTRY                   |9556560.453559974   |485.00611315265803|
|FROZEN FOODS              |820269

# **Interpretación**

Las categorías que aparecen en las primeras posiciones son aquellas que obtienen mejores resultados cuando existen promociones activas.

Este análisis demuestra cómo las promociones pueden influir en el comportamiento de compra de los clientes y ayudar a incrementar las ventas en determinadas categorías.

# **Pregunta 1**

El patrón que proporciona el hallazgo más importante para este proyecto es el Ranking con Window Function.

Este análisis permite identificar la categoría con mayores ventas dentro de cada tienda, información que resulta muy útil para los responsables comerciales al momento de gestionar inventarios, planificar promociones y distribuir productos de manera más eficiente.

Además, este patrón ofrece una visión más específica del comportamiento de cada tienda que un simple conteo o una agregación general.

In [10]:
top_categoria_por_tienda.explain(True)

== Parsed Logical Plan ==
'Filter '`=`('ranking, 1)
+- Project [store_nbr#105, family#106, ventas_totales#305, ranking#318]
   +- Project [store_nbr#105, family#106, ventas_totales#305, ranking#318, ranking#318]
      +- Window [row_number() windowspecdefinition(store_nbr#105, ventas_totales#305 DESC NULLS LAST, specifiedwindowframe(RowFrame, unboundedpreceding$(), currentrow$())) AS ranking#318], [store_nbr#105], [ventas_totales#305 DESC NULLS LAST]
         +- Project [store_nbr#105, family#106, ventas_totales#305]
            +- Aggregate [store_nbr#105, family#106], [store_nbr#105, family#106, sum(sales#107) AS ventas_totales#305]
               +- Project [date#104, store_nbr#105, id#103, family#106, sales#107, onpromotion#108, city#127, state#128, type#129, cluster#130, dcoilwtico#149]
                  +- Join LeftOuter, (date#104 = date#148)
                     :- Project [store_nbr#105, id#103, date#104, family#106, sales#107, onpromotion#108, city#127, state#128, type#129, c

# **Pregunta 2**
La operación más costosa computacionalmente fue el ranking utilizando Window Function.

Esto se debe a que Spark debe agrupar los datos, redistribuirlos entre particiones y ordenarlos antes de calcular la posición de cada registro dentro de su grupo.

Al observar el plan de ejecución mediante explain(), aparecen operaciones como Exchange, Sort y Window. La operación Exchange indica la existencia de un shuffle de datos.

El shuffle es costoso porque implica mover información entre particiones, aumentando el tiempo de procesamiento y el consumo de recursos del sistema.

# **Pregunta 3**

A partir de los resultados obtenidos y del análisis realizado en la actividad TA-2.2, se identifican varias transformaciones necesarias para la siguiente fase del proyecto.

En primer lugar, será necesario tratar los valores nulos presentes en la variable dcoilwtico mediante técnicas de imputación.

También será conveniente escalar variables numéricas como sales, onpromotion y dcoilwtico para mejorar el rendimiento de los modelos predictivos.

Además, las variables categóricas como family, city, state y type deberán ser codificadas para que puedan ser utilizadas por algoritmos de aprendizaje automático.

Finalmente, será importante revisar y tratar posibles valores atípicos en la variable sales para reducir el impacto de observaciones extremas en las predicciones futuras.